# Notebook 2 - Time Series Cluster Rasterization and Reclassification


This notebook convert shapefiles to raster and reclassifies into new classes from 1 to 6. The reclassification definitions can be found in Appendix A in the report in /results/reports...

## Import libraries and set up paths

In [1]:
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import geopandas as gpd
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Workspace-relative paths
REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT ).exists():
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = REPO_ROOT / 'data'
RAW_DIR = DATA_DIR  /'raw'
RESULTS_DIR = REPO_ROOT / 'results'

# Set workspace paths
WORKSPACE = str(REPO_ROOT)
TSC_FOLDER = os.fspath(DATA_DIR / 'processed' / 'time_series_cluster')
OUTPUT_FOLDER = os.fspath(DATA_DIR / 'processed' / 'tsc_reclass')
CLUSTERS_SHP = os.fspath(RAW_DIR / 'GIS_lag' / 'clusters_hovedstad_clean.shp')
RECLASS_EXCEL = os.fspath(DATA_DIR /'interim' / 'reclass_tsc.xlsx')

# Create output folder
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"Workspace: {WORKSPACE}")
print(f"TSC folder: {TSC_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")
print(f"\nChecking if files exist:")
print(f"  Clusters shapefile: {os.path.exists(CLUSTERS_SHP)}")
print(f"  Reclass Excel: {os.path.exists(RECLASS_EXCEL)}")

Workspace: C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model
TSC folder: C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\data\processed\time_series_cluster
Output folder: C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\data\processed\tsc_reclass

Checking if files exist:
  Clusters shapefile: True
  Reclass Excel: True


## Load reclassification rules from Excel

In [2]:
# Read Excel file with reclassification rules
reclass_df = pd.read_excel(RECLASS_EXCEL)
print("Reclassification table:")
print(reclass_df)
print(f"\nColumns: {list(reclass_df.columns)}")

# Column names from your Excel file
var_col = 'variable'  # Variable name (TSC_EMUB, TSC_PUB, etc.)
from_col = 'from'     # Input value (cluster ID)
to_col = 'to'         # Output value (reclassified)

print(f"\nUsing columns:")
print(f"  Variable: {var_col}")
print(f"  From: {from_col}")
print(f"  To: {to_col}")

# Extract variable names (remove 'TSC_' prefix to match shapefile names)
reclass_df['var_name'] = reclass_df[var_col].str.replace('TSC_', '')

# Create dictionary: variable_name -> {from_val: to_val}
reclass_rules = {}
for var in reclass_df['var_name'].unique():
    var_data = reclass_df[reclass_df['var_name'] == var]
    var_dict = dict(zip(var_data[from_col].astype(int), var_data[to_col].astype(int)))
    reclass_rules[var] = var_dict
    print(f"\n{var}: {var_dict}")

print(f"\n✓ Loaded {len(reclass_rules)} variable reclassification rules")


Reclassification table:
         variable  from  to
0        TSC_EMUB     1   3
1        TSC_EMUB     2   3
2        TSC_EMUB     3   3
3        TSC_EMUB     4   2
4        TSC_EMUB     5   2
..            ...   ...  ..
109  TSC_mean_sqm     2   2
110  TSC_mean_sqm     3   1
111  TSC_mean_sqm     4   1
112  TSC_mean_sqm     5   1
113  TSC_mean_sqm     6   3

[114 rows x 3 columns]

Columns: ['variable', 'from', 'to']

Using columns:
  Variable: variable
  From: from
  To: to

EMUB: {1: 3, 2: 3, 3: 3, 4: 2, 5: 2, 6: 3}

PMB: {1: 3, 2: 3, 3: 3, 4: 3, 5: 3, 6: 3}

PUB: {1: 4, 2: 3, 3: 6, 4: 6, 5: 4, 6: 3}

Age_18_25: {1: 2, 2: 4, 3: 1, 4: 3, 5: 4, 6: 6}

age_26_40: {1: 2, 2: 2, 3: 6, 4: 2, 5: 1, 6: 5}

age_41_55: {1: 2, 2: 1, 3: 1, 4: 2, 5: 3, 6: 6}

age_56_69: {1: 1, 2: 1, 3: 5, 4: 1, 5: 1, 6: 6}

crime_main_y: {1: 3, 2: 3, 3: 3, 4: 2, 5: 1, 6: 6}

disp_inc: {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6}

emp: {1: 2, 2: 5, 3: 2, 4: 4, 5: 5, 6: 1}

grund: {1: 2, 2: 3, 3: 1, 4: 1, 5: 2, 6: 5}

gym_e

## Get CRS and bounds from clusters shapefile

In [3]:
# Read clusters shapefile to get CRS and bounds
clusters_gdf = gpd.read_file(CLUSTERS_SHP)
print(f"Clusters CRS: {clusters_gdf.crs}")
print(f"Clusters bounds: {clusters_gdf.total_bounds}")

# Get bounds for rasterization
minx, miny, maxx, maxy = clusters_gdf.total_bounds
crs = clusters_gdf.crs
cell_size = 100  # 100 meter pixels

# IMPORTANT: Adjust bounds to align with cell grid
# Round down minx/miny and round up maxx/maxy to nearest cell_size
minx_aligned = np.floor(minx / cell_size) * cell_size
miny_aligned = np.floor(miny / cell_size) * cell_size
maxx_aligned = np.ceil(maxx / cell_size) * cell_size
maxy_aligned = np.ceil(maxy / cell_size) * cell_size

print(f"\nOriginal bounds: ({minx:.2f}, {miny:.2f}, {maxx:.2f}, {maxy:.2f})")
print(f"Aligned bounds:  ({minx_aligned:.2f}, {miny_aligned:.2f}, {maxx_aligned:.2f}, {maxy_aligned:.2f})")

# Use aligned bounds
minx, miny, maxx, maxy = minx_aligned, miny_aligned, maxx_aligned, maxy_aligned

# Calculate raster dimensions from aligned bounds
width = int((maxx - minx) / cell_size)
height = int((maxy - miny) / cell_size)

print(f"\nRaster dimensions: {width} x {height}")
print(f"Cell size: {cell_size} meters")
print(f"Raster area: {(maxx-minx):.0f} x {(maxy-miny):.0f} meters")

Clusters CRS: EPSG:25832
Clusters bounds: [ 716900. 6170600.  729700. 6182000.]

Original bounds: (716900.00, 6170600.00, 729700.00, 6182000.00)
Aligned bounds:  (716900.00, 6170600.00, 729700.00, 6182000.00)

Raster dimensions: 128 x 114
Cell size: 100 meters
Raster area: 12800 x 11400 meters


## Find all shapefiles in time_series_cluster folder

In [4]:
# Find all shapefiles (excluding _chart files)
shp_files = [f for f in os.listdir(TSC_FOLDER) if f.endswith('.shp') and '_chart' not in f]
shp_files = sorted(shp_files)

print(f"Found {len(shp_files)} shapefiles:")
for shp in shp_files:
    print(f"  - {shp}")

Found 23 shapefiles:
  - TSC_EMUB.shp
  - TSC_PMB.shp
  - TSC_PUB.shp
  - TSC_age_18_25.shp
  - TSC_age_26_40.shp
  - TSC_age_41_55.shp
  - TSC_age_56_69.shp
  - TSC_counts.shp
  - TSC_crime_main_y.shp
  - TSC_disp_inc.shp
  - TSC_emp.shp
  - TSC_grund.shp
  - TSC_gym_erhv.shp
  - TSC_lvu.shp
  - TSC_mean_price.shp
  - TSC_mean_sqm.shp
  - TSC_mig_in.shp
  - TSC_mig_net.shp
  - TSC_mig_out.shp
  - TSC_ool.shp
  - TSC_public_housing.shp
  - TSC_qol.shp
  - TSC_unemp.shp


## Helper function for reclassification

In [5]:
def reclassify_raster(raster_array, mapping_dict):
    """
    Reclassify raster values based on mapping dictionary
    """
    output = np.copy(raster_array).astype(float)
    output[:] = np.nan  # Initialize with NaN
    
    for from_val, to_val in mapping_dict.items():
        mask = raster_array == from_val
        output[mask] = to_val
    
    return output

print("Reclassification function defined!")

Reclassification function defined!


## Step 1 & 2: Convert and Reclassify

In [6]:
results = {}

for shp_file in shp_files:
    try:
        # Extract variable name
        var_name = shp_file.replace('TSC_', '').replace('.shp', '')
        print(f"\n{'='*60}")
        print(f"Processing: {var_name}")
        print(f"{'='*60}")
        
        shp_path = os.path.join(TSC_FOLDER, shp_file)
        
        # Step 1: Read shapefile and rasterize
        print(f"Step 1: Converting shapefile to raster...")
        gdf = gpd.read_file(shp_path)
        
        # Check if CLUSTER_ID exists
        if 'CLUSTER_ID' not in gdf.columns:
            print(f"  Warning: CLUSTER_ID not found. Available columns: {list(gdf.columns)}")
            cluster_col = [col for col in gdf.columns if 'cluster' in col.lower()]
            cluster_col = cluster_col[0] if cluster_col else gdf.columns[0]
        else:
            cluster_col = 'CLUSTER_ID'
        
        print(f"  Using column: {cluster_col}")
        
        # Create transform with ALIGNED bounds (from cell 6)
        # Top-left corner is minx, maxy (not miny!)
        transform = from_bounds(minx, miny, maxx, maxy, width, height)
        
        print(f"  Transform: {transform}")
        
        # Prepare geometries with values
        shapes = [(geom, int(value)) for geom, value in zip(gdf.geometry, gdf[cluster_col])]
        
        # Rasterize
        raster_array = rasterize(
            shapes,
            out_shape=(height, width),
            transform=transform,
            fill=0,
            dtype=rasterio.uint8
        )
        
        print(f"  ✓ Raster created: shape {raster_array.shape}")
        
        # Step 2: Reclassify - use variable-specific rules if available
        print(f"Step 2: Reclassifying raster...")
        
        if reclass_rules and var_name in reclass_rules:
            current_mapping = reclass_rules[var_name]
            print(f"  Using variable-specific rules: {current_mapping}")
        elif reclass_rules:
            # Try to find matching variable in rules
            matching = [k for k in reclass_rules.keys() if k.lower() == var_name.lower()]
            if matching:
                current_mapping = reclass_rules[matching[0]]
                print(f"  Using rules for: {matching[0]}")
            else:
                print(f"  Warning: No rules found for {var_name}. Available: {list(reclass_rules.keys())}")
                current_mapping = {}
        else:
            # Use common rules
            current_mapping = reclass_dict
            print(f"  Using common rules")
        
        reclassified = reclassify_raster(raster_array, current_mapping)
        
        # Handle NaN values
        reclassified = np.where(np.isnan(reclassified), 0, reclassified).astype(rasterio.uint8)
        
        print(f"  ✓ Reclassification complete")
        
        # Save reclassified raster
        output_raster = os.path.join(OUTPUT_FOLDER, f"reclass_{var_name}.tif")
        
        with rasterio.open(
            output_raster,
            'w',
            driver='GTiff',
            height=height,
            width=width,
            count=1,
            dtype=reclassified.dtype,
            crs=crs,
            transform=transform,
        ) as dst:
            dst.write(reclassified, 1)
        
        print(f"  ✓ Reclassified raster saved: {output_raster}")
        results[var_name] = "✓ Success"
        
    except Exception as e:
        print(f"  ✗ Error: {str(e)}")
        import traceback
        traceback.print_exc()
        results[var_name] = f"✗ Failed: {str(e)}"

print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
for var, status in results.items():
    print(f"{status}: {var}")


Processing: EMUB
Step 1: Converting shapefile to raster...
  Using column: CLUSTER_ID
  Transform: | 100.00, 0.00, 716900.00|
| 0.00,-100.00, 6182000.00|
| 0.00, 0.00, 1.00|
  ✓ Raster created: shape (114, 128)
Step 2: Reclassifying raster...
  Using variable-specific rules: {1: 3, 2: 3, 3: 3, 4: 2, 5: 2, 6: 3}
  ✓ Reclassification complete
  ✓ Reclassified raster saved: C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\data\processed\tsc_reclass\reclass_EMUB.tif

Processing: PMB
Step 1: Converting shapefile to raster...
  Using column: CLUSTER_ID
  Transform: | 100.00, 0.00, 716900.00|
| 0.00,-100.00, 6182000.00|
| 0.00, 0.00, 1.00|
  ✓ Raster created: shape (114, 128)
Step 2: Reclassifying raster...
  Using variable-specific rules: {1: 3, 2: 3, 3: 3, 4: 3, 5: 3, 6: 3}
  ✓ Reclassification complete
  ✓ Reclassified raster saved: C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\data\processed\tsc_reclass\reclass_PMB.tif

Processing: PUB
Step 1: Converting sh